# Colab: Export Gemma-4 E2B ASL LoRA to merged HF model and Cactus conversion path

This notebook takes the LoRA adapter created by `06_colab_gemma4_e2b_top50_q64_finetune.ipynb`, merges it with `unsloth/gemma-4-E2B-it`, uploads the merged model to Hugging Face, then provides a Cactus conversion cell.

Cactus docs support `cactus convert BASE_MODEL OUT_DIR --lora ADAPTER`, but for Gemma-4/Unsloth adapters the safer path from our project notes is:

1. Load the adapter repo directly with Unsloth.
2. `save_pretrained_merged(..., save_method="merged_16bit")`.
3. Upload merged HF model.
4. Run `cactus convert MERGED_REPO OUT_DIR --precision INT4` with **no `--lora` flag**.


In [ ]:
import os
if "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ:
    !pip install -U --no-cache-dir unsloth unsloth_zoo
    !pip install -U --no-cache-dir "transformers>=4.56.0" "peft>=0.17.0" accelerate bitsandbytes huggingface_hub
else:
    print("Not running in Colab; skipping install cell.")


In [ ]:
import os
from getpass import getpass
from pathlib import Path

import torch
from huggingface_hub import HfApi, login

hf_token = os.environ.get("HF_TOKEN") or getpass("HF_TOKEN: ")
login(token=hf_token, add_to_git_credential=False)
api = HfApi(token=hf_token)
print(api.whoami()["name"])
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)


In [ ]:
HF_NAMESPACE = "AlexD281"
BASE_MODEL = "unsloth/gemma-4-E2B-it"
ADAPTER_REPO_ID = f"{HF_NAMESPACE}/asl-gemma4-e2b-q64-top50-lora"
MERGED_REPO_ID = f"{HF_NAMESPACE}/asl-gemma4-e2b-q64-top50-merged-16bit"

WORK_DIR = Path("/content/asl_gemma4_e2b_export")
MERGED_DIR = WORK_DIR / "merged_16bit"
CACTUS_OUT_DIR = WORK_DIR / "cactus_int4"
WORK_DIR.mkdir(parents=True, exist_ok=True)

MAX_SEQ_LENGTH = 2048
print("adapter:", ADAPTER_REPO_ID)
print("merged:", MERGED_REPO_ID)


In [ ]:
# HF / Unsloth download stability preflight.
# Run this BEFORE importing `unsloth`.
# The 120s error you saw is Unsloth's optional environment/statistics ping,
# not the Gemma-4 model download. Disable that ping but keep Hugging Face
# enabled for the actual Gemma-4 model files.
import os

os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"
os.environ.pop("UNSLOTH_USE_MODELSCOPE", None)  # keep using Hugging Face, not ModelScope

# Longer HF Hub timeouts help Colab when the real model download is slow.
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "60")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "600")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")

print("Disabled Unsloth statistics check; Gemma-4 will still download from Hugging Face.")



In [ ]:
# Load the adapter repo through Unsloth. This avoids generic PEFT adapter injection issues seen with Gemma-4 adapters.
from unsloth import FastModel

try:
    model, tokenizer = FastModel.from_pretrained(
        model_name=ADAPTER_REPO_ID,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
        full_finetuning=False,
        token=hf_token,
    )
except TimeoutError as exc:
    raise RuntimeError(
        "Unsloth timed out before model loading. Restart the runtime, run the "
        "'HF / Unsloth download stability preflight' cell, confirm it prints "
        "that UNSLOTH_DISABLE_STATISTICS is enabled, then re-run this cell. "
        "This keeps the adapter/base model download on Hugging Face."
    ) from exc

print("loaded", ADAPTER_REPO_ID)
print("Tokenizer class:", type(tokenizer).__name__)



In [ ]:
if "model" not in globals() or "tokenizer" not in globals():
    raise RuntimeError("model/tokenizer are not defined. Run the previous adapter load cell first.")

# Smoke inference before merge.
FastModel.for_inference(model)
messages = [{"role": "user", "content": "Classify this compact ASL pose encoding into its WLASL gloss. Reply with only the gloss word.\n\nsample_id=smoke\nencoding=q64_full\nframes=0 features_per_frame=236\npose_q64="}]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    enable_thinking=False,
    return_tensors="pt",
    return_dict=True,
).to("cuda")
with torch.inference_mode():
    out = model.generate(**inputs, max_new_tokens=8, do_sample=False, use_cache=True)
print(tokenizer.decode(out[0, inputs["input_ids"].shape[-1]:], skip_special_tokens=True))


In [ ]:
if "model" not in globals() or "tokenizer" not in globals():
    raise RuntimeError("model/tokenizer are not defined. Run the adapter load cell before merging.")

# Merge adapter into a full 16-bit HF model directory.
# This is the artifact Cactus should convert without a --lora flag.
MERGED_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained_merged(str(MERGED_DIR), tokenizer, save_method="merged_16bit")
print("merged model saved:", MERGED_DIR)


In [ ]:
# Upload merged model to Hugging Face. Large upload can take a while.
api.create_repo(MERGED_REPO_ID, repo_type="model", private=True, exist_ok=True)
api.upload_large_folder(
    repo_id=MERGED_REPO_ID,
    repo_type="model",
    folder_path=str(MERGED_DIR),
)
print("uploaded merged model:", MERGED_REPO_ID)


## Cactus conversion

Run the next cell if you want to try Cactus conversion directly in Colab. If it fails due to Colab/Linux dependency differences, download or use the merged HF repo on your Linux GPU/CPU box or Mac with the Cactus repo already set up:

```bash
cd ~/cactus
source ./setup
cactus convert AlexD281/asl-gemma4-e2b-q64-top50-merged-16bit ./asl-gemma4-e2b-q64-top50-cactus-int4 --precision INT4
cactus build
cactus run ./asl-gemma4-e2b-q64-top50-cactus-int4
```


In [ ]:
RUN_CACTUS_CONVERT_IN_COLAB = False
if RUN_CACTUS_CONVERT_IN_COLAB:
    import os
    os.chdir("/content")
    !git clone https://github.com/cactus-compute/cactus || true
    os.chdir("/content/cactus")
    !bash -lc 'source ./setup && cactus convert {MERGED_REPO_ID} {CACTUS_OUT_DIR} --precision INT4'
    print("Cactus output:", CACTUS_OUT_DIR)


In [ ]:
# Optional: zip the merged model for manual download from Colab.
ZIP_MERGED_FOR_DOWNLOAD = False
if ZIP_MERGED_FOR_DOWNLOAD:
    import os
    os.chdir(WORK_DIR)
    !zip -r merged_16bit.zip merged_16bit
    from google.colab import files
    files.download(str(WORK_DIR / "merged_16bit.zip"))
